# 🔬 Multi-feature steering — with proper controls

Notebook 26 found a −15pp shift in unknown refusal at K=200 multi-feature ablation, but we published it without the canonical controls. This notebook adds the six controls the literature requires before claiming "circuit-level causal effect":

1. **Random-K ablation control** — R=30 random draws of K features, build null distribution. The single most important control. Ferrando 2024's EA Forum follow-up showed their 4,697 entity-feature ablation (−30pp) was matched by 4,697 *random* features (−29.2pp). Our K=200 of 65k latents (~0.3%) is in the same danger zone.
2. **Direction-sorted sweep** — separate top-K positive-d (fires-on-known) from top-K negative-d (fires-on-unknown). Mixed |d| in notebook 26 may have cancelled effects across directions.
3. **Anti-feature control** — bottom-K |d| should give ≈0 effect. If it gives the same effect as top-K, top-K isn't selecting anything useful.
4. **3-way split** — select features on TRAIN, tune K on VAL, report effect on TEST. Notebook 26 used same prompts for selection AND eval (selection bias ~15pp per our own memory note `feedback_feature_selection_leakage.md`).
5. **Confabulation-vs-correct labelling** — for each "refusal lost" generation, LLM-judge whether the answer is correct. Distinguishes "intervention reduced hedging" from "intervention induced more hallucination".
6. **Permutation test on Cohen's d** — permute known/unknown labels 1000×, recompute top-K each time, build null effect-size distribution. CPU-only, no extra GPU.

**Three possible outcomes**:
- ✅ Random-K = null + direction-sorted = clean: circuit story validated
- ⚠️ Top-K beats random by ≥5pp at p<0.05: weak but defensible
- ❌ Random-K matches top-K: finding collapses, honest negative writeup

**Cost**: ~6-8 GPU hours on RTX 6000 Pro. ~$20. Self-contained (runs from fresh Colab kernel).

References:
- [Ferrando 2024 — Do I Know This Entity? (arXiv:2411.14257)](https://arxiv.org/abs/2411.14257)
- [EA Forum follow-up — random-control failure](https://forum.effectivealtruism.org/posts/9vkEpghkuEv5QtfKG/entity-recognition-feature-steering-in-gemma-2-2b)
- [Li & Janson 2024 — Optimal ablation (arXiv:2409.09951)](https://arxiv.org/abs/2409.09951)

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub datasets matplotlib tqdm requests scipy
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 1. Config + load model + L11 SAE

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
STEER_LAYER   = 11
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

K_SWEEP        = [0, 5, 20, 50, 200]
RANDOM_DRAWS   = 30        # R=30 per K for random control
PER_TYPE_CAND  = 200       # over-sample for 3-way split
N_TARGET_PER_CLASS = 60    # split 60 → 30 select / 15 tune / 15 report
PILE_THRESHOLD = 0.02
N_PILE_TOKENS  = 2000
MAX_GEN_TOKENS = 80
JUDGE_MODEL    = 'anthropic/claude-haiku-4.5'   # cheap LLM judge for confabulation labelling

import os, math, json, time, random, re
import numpy as np
import requests
from collections import Counter
from datetime import datetime, timezone
from itertools import product
random.seed(0); torch.manual_seed(0); np.random.seed(0)

from huggingface_hub import login, hf_hub_download, HfApi
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    OPENROUTER_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    login()
    OPENROUTER_KEY = os.environ.get('OPENROUTER_API_KEY') or input('OPENROUTER_API_KEY: ')

from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL, dtype=torch.bfloat16, attn_implementation='sdpa',
    device_map='cuda', trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z
    def decode(self, z):
        return z @ self.W_dec + self.b_dec

sae = TopKSAE(load_file(hf_hub_download(HF_SAE_REPO, f'sae_L{STEER_LAYER}_latest.safetensors')), K).to(device).eval()
layer_mod = model.model.language_model.layers[STEER_LAYER]
print(f'  ✓ ready · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 2. Pull entities + label larger pool (target ≥60 per class)

In [ ]:
from tqdm.auto import tqdm

FERRANDO_BASE = 'https://raw.githubusercontent.com/javiferran/sae_entities/main/dataset/processed/entities'
ENTITY_TYPES = ['player', 'movie']
raw = {t: requests.get(f'{FERRANDO_BASE}/{t}.json', timeout=60).json() for t in ENTITY_TYPES}
rng = random.Random(0)
candidates = []
for t in ENTITY_TYPES:
    sample = rng.sample(raw[t], min(PER_TYPE_CAND, len(raw[t])))
    for ent in sample:
        candidates.append({'type': t, **ent})
print(f'candidates: {len(candidates)}')

ATTRS = {
    'player': {
        'place_birth':  "What is the place of birth of the basketball player '{entity}'? Answer in just one or two words.",
        'date_birth':   "In what year was the basketball player '{entity}' born? Answer with just a year.",
        'teams_list':   "What was a team that the basketball player '{entity}' played for? Answer with just the team name.",
    },
    'movie': {
        'directors':    "Who directed the movie '{entity}'? Answer with just the director's name.",
        'release_year': "In what year was the movie '{entity}' released? Answer with just a year.",
        'genres':       "What is one genre of the movie '{entity}'? Answer with just one word.",
    },
}

REFUSAL_RE = re.compile('|'.join([
    r"i (?:don'?t|do not) (?:know|have)",
    r"i'?m (?:sorry|not sure|not familiar|unable)",
    r"i (?:cannot|can'?t) (?:provide|verify|confirm|find)",
    r"there (?:is|seems to be) (?:no|insufficient|limited) (?:information|data|record)",
    r"unable to (?:find|locate|verify)",
    r"(?:no|not enough|insufficient) (?:public(?:ly available)?\s+)?(?:information|data|record)",
    r"i don't have (?:specific|enough|reliable) (?:information|details|data)",
    r"there'?s no (?:widely|publicly|reliable) (?:known|available)",
    r"no widely recognized (?:public figure|professional|celebrity|historical figure)",
]), re.IGNORECASE)
def is_refusal(t): return bool(REFUSAL_RE.search(t or ''))
def normalise(s): return re.sub(r'[^a-z0-9]+', ' ', (s or '').lower()).strip()
def attr_match(a, gt):
    if isinstance(gt, list): return any(attr_match(a, x) for x in gt)
    if not gt or not a: return False
    A, G = normalise(str(a)), normalise(str(gt))
    return bool(G) and (G in A or A in G)

def chat_short(q):
    msgs = [
        {'role': 'system', 'content': 'Answer concisely and directly. Do not show reasoning. If you do not know, say "I do not know".'},
        {'role': 'user', 'content': q},
    ]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(ids['input_ids'], attention_mask=ids['attention_mask'],
                              max_new_tokens=80, do_sample=False, pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    if 'thinking process' in ans.lower():
        parts = [p for p in ans.split('\n') if p.strip()]
        ans = parts[-1] if parts else ans
    return ans

labelled = []
for ent in tqdm(candidates, desc='label'):
    name, type_ = ent['entity'], ent['type']
    gt = {}
    for a in ent.get('attributes', []):
        gt.setdefault(a['attribute_type'], []).append(a['attribute_value'])
    avail = [k for k in ATTRS[type_] if k in gt]
    if len(avail) < 2: continue
    selected = avail[:3]
    nc, nr = 0, 0
    for at in selected:
        a = chat_short(ATTRS[type_][at].format(entity=name))
        if is_refusal(a): nr += 1
        elif attr_match(a, gt[at]): nc += 1
    if nc >= 2 and nr == 0: cls = 'known'
    elif nc == 0 and nr >= 1: cls = 'unknown'
    else: cls = 'middle'
    labelled.append({'type': type_, 'entity': name, 'class': cls, 'attributes': ent['attributes']})
    c = Counter(l['class'] for l in labelled)
    if c.get('known', 0) >= N_TARGET_PER_CLASS and c.get('unknown', 0) >= N_TARGET_PER_CLASS:
        break

known_all   = [l for l in labelled if l['class'] == 'known']
unknown_all = [l for l in labelled if l['class'] == 'unknown']
print(f'\nlabelled: {len(known_all)} known + {len(unknown_all)} unknown')

# 3-way split: select / tune / report (60/20/20)
rng2 = random.Random(0)
rng2.shuffle(known_all); rng2.shuffle(unknown_all)
def split3(lst):
    n = len(lst)
    a, b = int(0.6*n), int(0.8*n)
    return lst[:a], lst[a:b], lst[b:]
k_sel, k_tune, k_rep = split3(known_all)
u_sel, u_tune, u_rep = split3(unknown_all)
print(f'  select: {len(k_sel)} k + {len(u_sel)} u')
print(f'  tune:   {len(k_tune)} k + {len(u_tune)} u')
print(f'  report: {len(k_rep)} k + {len(u_rep)} u')

## 3. Capture activations on SELECT split → compute Cohen's d → Pile filter

Critical: feature ranking is built from SELECT only. TUNE and REPORT splits never inform the ranking.

In [ ]:
from datasets import load_dataset

PROMPT_TPL = "What can you tell me about '{entity}'?"
_cap = {}
def cap_hook(mod, inp, out):
    _cap['h'] = out[0] if isinstance(out, tuple) else out
    return out
def find_pos(prompt):
    ids_ = tok(prompt, return_tensors='pt')['input_ids'][0].tolist()
    cq = tok.encode("'?", add_special_tokens=False)
    for i in range(len(ids_)-len(cq), -1, -1):
        if ids_[i:i+len(cq)] == cq: return i-1
    return len(ids_)-2

def encode_set(entries, label):
    zs = []
    h = layer_mod.register_forward_hook(cap_hook)
    with torch.no_grad():
        for ent in tqdm(entries, desc=label):
            prompt = PROMPT_TPL.format(entity=ent['entity'])
            ids = tok(prompt, return_tensors='pt')['input_ids'].to(device)
            pos = find_pos(prompt)
            _ = model(ids)
            resid = _cap['h'][0, pos].to(torch.bfloat16)
            zs.append(sae.encode(resid.unsqueeze(0))[0].float().cpu().numpy())
    h.remove()
    return np.stack(zs, axis=0)

Z_k_sel = encode_set(k_sel, 'select known')
Z_u_sel = encode_set(u_sel, 'select unknown')

# Pile filter
pile_ds = load_dataset('NeelNanda/pile-10k', split='train', streaming=True)
pt = []
for x in pile_ds:
    pt.append(x['text'])
    if len(pt) >= 50: break
pile_ids = tok(' '.join(pt), return_tensors='pt', max_length=N_PILE_TOKENS, truncation=True)['input_ids'].to(device)
h = layer_mod.register_forward_hook(cap_hook)
with torch.no_grad(): _ = model(pile_ids)
h.remove()
pile_z = sae.encode(_cap['h'][0].to(torch.bfloat16))
pile_fr = (pile_z > 0).float().mean(0).cpu().numpy()
print(f'Pile: {(pile_fr > PILE_THRESHOLD).sum()} features dropped')

# Cohen's d, signed
mu_k, mu_u = Z_k_sel.mean(0), Z_u_sel.mean(0)
sd = np.sqrt((Z_k_sel.std(0)**2 + Z_u_sel.std(0)**2) / 2) + 1e-9
sep = (mu_k - mu_u) / sd
sep_filtered = sep.copy()
sep_filtered[pile_fr > PILE_THRESHOLD] = 0

# Three rankings: top-K |d|, top-K positive d (fires-on-known), top-K negative d (fires-on-unknown)
rank_abs = np.argsort(np.abs(sep_filtered))[::-1]   # mixed direction
rank_pos = np.argsort(-sep_filtered)                 # most positive first (fires-on-known)
rank_neg = np.argsort(sep_filtered)                  # most negative first (fires-on-unknown)
rank_bot = np.argsort(np.abs(sep_filtered))          # bottom-|d| (anti-feature control)

print('\nTop 5 by each ranking (post Pile filter):')
for name, r in [('|d|', rank_abs), ('pos d', rank_pos), ('neg d', rank_neg), ('bottom |d|', rank_bot)]:
    feats = r[:5].tolist()
    seps = [f'{sep[f]:+.2f}' for f in feats]
    print(f'  {name:12s}: {feats}  ({seps})')

## 4. Multi-feature ablation hook (parametric over feature list)

In [ ]:
_active = {'feats': []}
def ablate_hook(mod, inp, out):
    if not _active['feats']:
        return out
    h = out[0] if isinstance(out, tuple) else out
    orig = h.dtype
    flat = h.reshape(-1, D_MODEL).to(torch.bfloat16)
    z = sae.encode(flat)
    recon0 = sae.decode(z).to(torch.float32)
    err = flat.to(torch.float32) - recon0
    z_mod = z.clone()
    for f in _active['feats']:
        z_mod[:, f] = 0
    recon1 = sae.decode(z_mod).to(torch.float32)
    new = (recon1 + err).to(orig).reshape(h.shape)
    if isinstance(out, tuple): return (new,) + out[1:]
    return new

_steer = layer_mod.register_forward_hook(ablate_hook)

def gen_with(feats, entity_name):
    _active['feats'] = list(feats)
    msgs = [{'role': 'user', 'content': f"What can you tell me about {entity_name}? Be specific."}]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(ids['input_ids'], attention_mask=ids['attention_mask'],
                              max_new_tokens=MAX_GEN_TOKENS, do_sample=False, pad_token_id=tok.eos_token_id)
    _active['feats'] = []
    return tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# Helper: refusal rate over an entity list × feature list
def refusal_rate(entries, feats):
    n = 0; r = 0
    for e in entries:
        txt = gen_with(feats, e['entity'])
        if is_refusal(txt): r += 1
        n += 1
    return r / n if n else 0.0

## 5. Main sweep on REPORT split: 4 ranking schemes × K sweep

Eval entities: REPORT split (never used to rank features). For each (ranking, K), record refusal rate on known and unknown.

We do NOT include random-K here — that's the next cell, with R=30 draws per K.

In [ ]:
main_sweep = {}
rankings = {'top_abs': rank_abs, 'top_pos_d': rank_pos, 'top_neg_d': rank_neg, 'bottom_abs': rank_bot}

all_eval_rep = [(e, 'known') for e in k_rep] + [(e, 'unknown') for e in u_rep]

# Cache baseline (K=0) once
print('Baseline (K=0) on REPORT split:')
baseline_texts = {}
for e, c in tqdm(all_eval_rep, desc='baseline'):
    baseline_texts[e['entity']] = gen_with([], e['entity'])
baseline_known   = sum(1 for e in k_rep if is_refusal(baseline_texts[e['entity']])) / max(len(k_rep),1)
baseline_unknown = sum(1 for e in u_rep if is_refusal(baseline_texts[e['entity']])) / max(len(u_rep),1)
print(f'  baseline known refusal:   {baseline_known:.1%}')
print(f'  baseline unknown refusal: {baseline_unknown:.1%}')

for rname, ranking in rankings.items():
    main_sweep[rname] = {0: {'known': baseline_known, 'unknown': baseline_unknown}}
    for K in K_SWEEP[1:]:
        feats = ranking[:K].tolist()
        rk = sum(1 for e in tqdm(k_rep, desc=f'{rname} K={K} known', leave=False)
                  if is_refusal(gen_with(feats, e['entity']))) / max(len(k_rep),1)
        ru = sum(1 for e in tqdm(u_rep, desc=f'{rname} K={K} unknown', leave=False)
                  if is_refusal(gen_with(feats, e['entity']))) / max(len(u_rep),1)
        main_sweep[rname][K] = {'known': rk, 'unknown': ru}

print('\n═══ Main sweep results (REPORT split, refusal rate) ═══')
for rname in rankings:
    print(f'\n{rname}:')
    print(f'         ' + '   '.join(f'K={k:>3d}' for k in K_SWEEP))
    print(f'  known  ' + '   '.join(f'{main_sweep[rname][k]["known"]:>5.1%}' for k in K_SWEEP))
    print(f'  unkn   ' + '   '.join(f'{main_sweep[rname][k]["unknown"]:>5.1%}' for k in K_SWEEP))

## 6. Random-K control — R=30 draws per K, build null distribution

**Critical control**. For each K ∈ {5, 20, 50, 200}, draw R=30 random sets of K features (uniform over Pile-passing latents, since dead features can't contribute) and measure refusal rate. Compare top-K effect against the 95th percentile of this null distribution.

In [ ]:
alive_features = np.where(pile_fr <= PILE_THRESHOLD)[0]
print(f'Pile-passing features available for random draws: {len(alive_features)}')

random_null = {K: {'known': [], 'unknown': []} for K in K_SWEEP[1:]}
rng3 = np.random.default_rng(0)

for K in K_SWEEP[1:]:
    for r in tqdm(range(RANDOM_DRAWS), desc=f'random K={K}'):
        feats = rng3.choice(alive_features, size=K, replace=False).tolist()
        rk = sum(1 for e in k_rep if is_refusal(gen_with(feats, e['entity']))) / max(len(k_rep),1)
        ru = sum(1 for e in u_rep if is_refusal(gen_with(feats, e['entity']))) / max(len(u_rep),1)
        random_null[K]['known'].append(rk)
        random_null[K]['unknown'].append(ru)

print('\n═══ Random-K null distribution (mean ± std over R=30 draws) ═══')
print(f'                    K=5             K=20            K=50            K=200')
for cls in ['known', 'unknown']:
    cells = []
    for K in K_SWEEP[1:]:
        m, s = float(np.mean(random_null[K][cls])), float(np.std(random_null[K][cls]))
        cells.append(f'{m:>5.1%} ± {s:>4.1%}')
    print(f'  {cls:9s}        ' + '   '.join(cells))

## 7. Compare top-K vs random-K — does top-K beat the null?

In [ ]:
from scipy.stats import percentileofscore

print('═' * 80)
print('Δ refusal rate on UNKNOWN (vs K=0 baseline)')
print('═' * 80)
print(f'{"":12s}  {"top |d|":>10s} {"top pos d":>10s} {"top neg d":>10s} {"bottom |d|":>10s} {"random mean":>12s} {"random p95":>11s}')
for K in K_SWEEP[1:]:
    cells = []
    for rname in ['top_abs', 'top_pos_d', 'top_neg_d', 'bottom_abs']:
        delta = main_sweep[rname][K]['unknown'] - baseline_unknown
        cells.append(f'{delta:+>9.1%}')
    rand = np.array(random_null[K]['unknown'])
    rand_d = rand - baseline_unknown
    cells.append(f'{rand_d.mean():+>11.1%}')
    cells.append(f'{np.percentile(rand_d, 5):+>10.1%}')   # 5th percentile (most negative)
    print(f'  K={K:>3d}      {"  ".join(cells)}')

print('\n═══ Verdict per ranking scheme @ K=200 ═══')
for rname in ['top_abs', 'top_pos_d', 'top_neg_d', 'bottom_abs']:
    K = 200
    obs_delta = main_sweep[rname][K]['unknown'] - baseline_unknown
    null = np.array(random_null[K]['unknown']) - baseline_unknown
    pct = percentileofscore(null, obs_delta, kind='strict')
    if obs_delta < null.min() and obs_delta < -0.05:
        verdict = '✓ exceeds null minimum, real effect'
    elif pct < 5:
        verdict = f'✓ p<0.05 ({pct:.1f}th pct), real effect'
    elif pct < 20:
        verdict = f'~ p~{pct:.0f}/100, weak'
    else:
        verdict = f'✗ p={pct:.0f}/100 — within random null'
    print(f'  {rname:14s} Δ={obs_delta:+.1%}  vs  null mean={null.mean():+.1%}, std={null.std():.1%}  →  {verdict}')

# Save raw
results = {
    'baseline_known':   baseline_known,
    'baseline_unknown': baseline_unknown,
    'main_sweep':       {r: {str(k): v for k, v in main_sweep[r].items()} for r in main_sweep},
    'random_null':      {str(k): random_null[k] for k in random_null},
    'rankings_top10': {
        'top_abs':    rank_abs[:10].tolist(),
        'top_pos_d':  rank_pos[:10].tolist(),
        'top_neg_d':  rank_neg[:10].tolist(),
        'bottom_abs': rank_bot[:10].tolist(),
    },
    'splits': {
        'select':  {'known': len(k_sel),  'unknown': len(u_sel)},
        'tune':    {'known': len(k_tune), 'unknown': len(u_tune)},
        'report':  {'known': len(k_rep),  'unknown': len(u_rep)},
    },
    'pile_threshold':       PILE_THRESHOLD,
    'random_draws_per_K':   RANDOM_DRAWS,
    'K_sweep':              K_SWEEP,
    'timestamp':            datetime.now(timezone.utc).isoformat(),
}

## 8. Confabulation-vs-correct labelling via Claude Haiku

For every generation that did NOT refuse, ask Claude to judge whether the answer is correct vs hallucinated. Distinguishes "intervention removed hedging" from "intervention induced more hallucination".

In [ ]:
import requests as _r

def judge_correct(entity, answer, attributes):
    """Returns 'correct' / 'incorrect' / 'unverifiable' from Claude judge."""
    attr_lines = '\n'.join(f'- {a["attribute_type"]}: {a["attribute_value"]}' for a in attributes[:5])
    prompt = (
        f"Wikidata ground truth for entity '{entity}':\n{attr_lines}\n\n"
        f"Model's answer: {answer[:600]}\n\n"
        f"Is the model's answer factually consistent with the ground truth? Reply ONLY one word: "
        f"'correct' (no contradictions with ground truth), 'incorrect' (contradicts ground truth), "
        f"or 'unverifiable' (vague enough to be either)."
    )
    try:
        r = _r.post('https://openrouter.ai/api/v1/chat/completions',
                    headers={'Authorization': f'Bearer {OPENROUTER_KEY}', 'Content-Type': 'application/json'},
                    json={'model': JUDGE_MODEL, 'max_tokens': 8, 'temperature': 0.0,
                          'messages': [{'role': 'user', 'content': prompt}]},
                    timeout=30)
        if r.status_code == 200:
            text = r.json()['choices'][0]['message'].get('content', '').strip().lower()
            for label in ['incorrect', 'correct', 'unverifiable']:
                if label in text: return label
    except Exception:
        pass
    return 'unverifiable'

# Judge: baseline + top_neg_d at K=200 (the most likely-to-have-signal config)
judge_results = {'baseline': {'known': [], 'unknown': []},
                 'top_neg_d_K200': {'known': [], 'unknown': []}}

rep_lookup = {e['entity']: e for e in k_rep + u_rep}
for e in tqdm(k_rep + u_rep, desc='judge baseline'):
    cls = 'known' if e in k_rep else 'unknown'
    txt = baseline_texts[e['entity']]
    if not is_refusal(txt):
        verdict = judge_correct(e['entity'], txt, e.get('attributes', []))
        judge_results['baseline'][cls].append({'entity': e['entity'], 'verdict': verdict})

_active['feats'] = rank_neg[:200].tolist()
for e in tqdm(k_rep + u_rep, desc='judge top_neg_d K=200'):
    cls = 'known' if e in k_rep else 'unknown'
    txt = gen_with(rank_neg[:200].tolist(), e['entity'])
    if not is_refusal(txt):
        verdict = judge_correct(e['entity'], txt, e.get('attributes', []))
        judge_results['top_neg_d_K200'][cls].append({'entity': e['entity'], 'verdict': verdict, 'text': txt[:200]})

print('\n═══ Confabulation-vs-correct (only non-refusal generations) ═══')
for cond_name, cond in judge_results.items():
    print(f'\n{cond_name}:')
    for cls in ['known', 'unknown']:
        items = cond[cls]
        if not items:
            print(f'  {cls}: no non-refusal generations')
            continue
        n = len(items)
        correct = sum(1 for x in items if x['verdict'] == 'correct')
        incorrect = sum(1 for x in items if x['verdict'] == 'incorrect')
        unverif = sum(1 for x in items if x['verdict'] == 'unverifiable')
        print(f'  {cls:8s} n={n:>3d}  correct={correct/n:.0%}  incorrect={incorrect/n:.0%}  unverifiable={unverif/n:.0%}')

results['judge'] = judge_results

## 9. Permutation test on Cohen's d ranking (CPU only)

Permute known/unknown labels 1000×, recompute Cohen's d, recompute top-K. How often does the *random* top-K under permuted labels include features in the top-K of our real ranking? If the overlap is high, our real ranking isn't selecting concept-specific features.

In [ ]:
Z_combined = np.concatenate([Z_k_sel, Z_u_sel], axis=0)
labels_real = np.concatenate([np.ones(len(Z_k_sel)), np.zeros(len(Z_u_sel))])
n_perm = 1000
rng4 = np.random.default_rng(0)

real_top200_pos = set(rank_pos[:200].tolist())
real_top200_neg = set(rank_neg[:200].tolist())
real_top200_abs = set(rank_abs[:200].tolist())

perm_overlaps = {'pos': [], 'neg': [], 'abs': []}
for _ in tqdm(range(n_perm), desc='permutation'):
    p_labels = rng4.permutation(labels_real)
    z_p_k = Z_combined[p_labels == 1]
    z_p_u = Z_combined[p_labels == 0]
    mu_p = z_p_k.mean(0) - z_p_u.mean(0)
    sd_p = np.sqrt((z_p_k.std(0)**2 + z_p_u.std(0)**2)/2) + 1e-9
    sep_p = mu_p / sd_p
    sep_p[pile_fr > PILE_THRESHOLD] = 0
    perm_pos = set(np.argsort(-sep_p)[:200].tolist())
    perm_neg = set(np.argsort(sep_p)[:200].tolist())
    perm_abs = set(np.argsort(np.abs(sep_p))[::-1][:200].tolist())
    perm_overlaps['pos'].append(len(perm_pos & real_top200_pos))
    perm_overlaps['neg'].append(len(perm_neg & real_top200_neg))
    perm_overlaps['abs'].append(len(perm_abs & real_top200_abs))

print('\nPermutation test (1000× shuffled labels) — overlap with real top-200:')
for kind, overlaps in perm_overlaps.items():
    overlaps = np.array(overlaps)
    print(f'  {kind:6s}: mean overlap={overlaps.mean():.1f}/200  median={np.median(overlaps):.0f}  p95={np.percentile(overlaps, 95):.0f}')
    print(f'           expected by chance: 200·(200/{(pile_fr<=PILE_THRESHOLD).sum()})={200*200/(pile_fr<=PILE_THRESHOLD).sum():.1f}')

results['permutation_overlaps'] = {k: list(map(int, v)) for k, v in perm_overlaps.items()}

## 10. Final verdict + save artifact

In [ ]:
print('═' * 80)
print('FINAL VERDICT — multi-feature SAE ablation with controls')
print('═' * 80)

K = 200
lines = []
any_strong = False
for rname in ['top_abs', 'top_pos_d', 'top_neg_d', 'bottom_abs']:
    obs = main_sweep[rname][K]['unknown'] - baseline_unknown
    null = np.array(random_null[K]['unknown']) - baseline_unknown
    pct = percentileofscore(null, obs, kind='strict')
    margin_vs_null_mean = obs - null.mean()
    is_real = (pct < 5) and (obs <= -0.05)
    if is_real:
        any_strong = True
    lines.append(f'  {rname:14s}  Δ={obs:+5.1%}  null={null.mean():+5.1%}±{null.std():4.1%}  pct={pct:>5.1f}  {"REAL" if is_real else "NOT-DISTINGUISHABLE"}')

for line in lines:
    print(line)

if any_strong:
    overall = 'AT LEAST ONE RANKING SHOWS REAL EFFECT'
    flag = '✅'
else:
    overall = 'ALL RANKINGS WITHIN RANDOM NULL — finding collapses to generic perturbation'
    flag = '❌'

print(f'\n{flag}  OVERALL: {overall}')

results['final_verdict'] = overall
results['any_ranking_significant'] = bool(any_strong)

# Save
with open('/tmp/multi_steering_with_controls.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

api = HfApi()
api.upload_file(
    path_or_fileobj='/tmp/multi_steering_with_controls.json',
    path_in_repo='multi_feature_steering_v0_0_2.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'Multi-feature steering v0.0.2 (with controls) — {overall[:60]}',
)
print(f'\n✓ uploaded · HF: multi_feature_steering_v0_0_2.json')